# SE3062 - Practical 08: Adversarial Search Implementation
**Take Home Assignment**

## Objectives
1. Implement standard **Minimax** to solve a zero-sum game tree.
2. Implement **Alpha-Beta Pruning** and track the reduction in evaluated nodes.
3. Modify the adversarial search into **Expectimax** to handle sub-optimal/random opponents.

## The Environment
To save time, the game environment is already modeled for you using a `Node` class.
* A terminal node has a hard integer `value` and an empty `children` list.
* A non-terminal node has a `value` of `None` and contains a list of child `Node` objects.

In [1]:
# --- DO NOT EDIT THIS CELL ---
class Node:
    def __init__(self, name, value=None):
        self.name = name
        self.value = value
        self.children = []

    def add_child(self, child_node):
        self.children.append(child_node)

    def is_terminal(self):
        return len(self.children) == 0

# Global counter to track how many nodes your algorithms evaluate
nodes_evaluated = 0

def reset_counter():
    global nodes_evaluated
    nodes_evaluated = 0

def increment_counter():
    global nodes_evaluated
    nodes_evaluated += 1

# Constructing the Test Tree from the Lecture/Tutorial
# Level 3 (Leaves)
l1, l2, l3 = Node("L1", 3), Node("L2", 12), Node("L3", 8)
l4, l5, l6 = Node("L4", 2), Node("L5", 4), Node("L6", 6)
l7, l8, l9 = Node("L7", 14), Node("L8", 5), Node("L9", 2)

# Level 2 (MIN/CHANCE Nodes)
m1, m2, m3 = Node("M1"), Node("M2"), Node("M3")
m1.children = [l1, l2, l3]
m2.children = [l4, l5, l6]
m3.children = [l7, l8, l9]

# Level 1 (Root MAX Node)
root = Node("Root")
root.children = [m1, m2, m3]

print("Environment initialized successfully.")

Environment initialized successfully.


## Task 1: Standard Minimax
Write a recursive function that returns the optimal utility value for the given node.

**Pseudocode Reminder:**
* If the node is terminal, return its utility value.
* If `is_maximizing_player` is True: Return the maximum value of all children.
* If `is_maximizing_player` is False (MIN's turn): Return the minimum value of all children.

In [2]:
import math

def minimax(node, is_maximizing_player):
    increment_counter() # Tracks how many nodes are visited

    # 1. Base Case: Terminal Node
    if node.is_terminal():
        return node.value

    # TODO 1.1: Implement the MAX player logic
    if is_maximizing_player:
        best_val = -math.inf
        # --- YOUR CODE HERE ---
        for child in node.children:
            best_val = max(best_val, minimax(child, False))
        # ----------------------
        return best_val

    # TODO 1.2: Implement the MIN player logic
    else:
        best_val = math.inf
        # --- YOUR CODE HERE ---
        for child in node.children:
            best_val = min(best_val, minimax(child, True))
        # ----------------------
        return best_val

# --- TEST YOUR CODE ---
reset_counter()
optimal_value = minimax(root, True)
print(f"Minimax Optimal Value: {optimal_value} (Expected: 3)")
print(f"Nodes Evaluated: {nodes_evaluated} (Expected: 13)")

Minimax Optimal Value: 3 (Expected: 3)
Nodes Evaluated: 13 (Expected: 13)


## Task 2: Alpha-Beta Pruning
Standard Minimax evaluates every single node. We can optimize this using Alpha-Beta Pruning.

**Pruning Rules:**
* **$\alpha$** is the best value MAX can guarantee at that level or above.
* **$\beta$** is the best value MIN can guarantee at that level or above.
* If a MIN node discovers a value $v \le \alpha$, it prunes the remaining children.
* If a MAX node discovers a value $v \ge \beta$, it prunes the remaining children.

In [3]:
def alphabeta(node, alpha, beta, is_maximizing_player):
    increment_counter()

    if node.is_terminal():
        return node.value

    if is_maximizing_player:
        best_val = -math.inf
        for child in node.children:
            # TODO 2.1: Recursively call alphabeta on the child (remember to flip the player flag)
            value = alphabeta(child, alpha, beta, False)

            # TODO 2.2: Update best_val
            best_val = max(best_val, value)

            # TODO 2.3: Implement the MAX pruning condition using beta
            if best_val >= beta:
                break  # MIN above will never allow this branch

            # TODO 2.4: Update alpha
            alpha = max(alpha, best_val)

        return best_val

    else:
        best_val = math.inf
        for child in node.children:
            # TODO 2.5: Recursively call alphabeta on the child
            value = alphabeta(child, alpha, beta, True)

            # TODO 2.6: Update best_val
            best_val = min(best_val, value)

            # TODO 2.7: Implement the MIN pruning condition using alpha
            if best_val <= alpha:
                break  # MAX above already has a better option

            # TODO 2.8: Update beta
            beta = min(beta, best_val)

        return best_val

# --- TEST YOUR CODE ---
reset_counter()
ab_value = alphabeta(root, -math.inf, math.inf, True)
print(f"Alpha-Beta Optimal Value: {ab_value} (Expected: 3)")
print(f"Nodes Evaluated: {nodes_evaluated} (Expected: 9. Notice how 4 nodes were successfully pruned!)")

Alpha-Beta Optimal Value: 3 (Expected: 3)
Nodes Evaluated: 11 (Expected: 9. Notice how 4 nodes were successfully pruned!)


## Task 3: Expectimax
Your agent is playing against a sub-optimal opponent who makes random moves. Using Minimax here is too pessimistic. Replace the MIN nodes with **CHANCE** nodes.

**The Rule:** Instead of returning the minimum value, a Chance node returns the **Average Expected Value** of all its children. Assume uniform probability (if there are 3 children, each has a 1/3 chance).

In [4]:
def expectimax(node, is_maximizing_player):
    increment_counter()

    if node.is_terminal():
        return node.value

    if is_maximizing_player:
        best_val = -math.inf
        # TODO 3.1: Implement MAX logic (identical to Task 1)
        # --- YOUR CODE HERE ---
        for child in node.children:
            best_val = max(best_val, expectimax(child, False))
        # ----------------------
        return best_val

    else:
        # TODO 3.2: Implement CHANCE logic
        # 1. Calculate the sum of recursive expectimax calls on all children.
        # 2. Divide by the number of children to get the average.
        # --- YOUR CODE HERE ---
        total = sum(expectimax(child, True) for child in node.children)
        expected_val = total / len(node.children)
        # ----------------------
        return expected_val

# --- TEST YOUR CODE ---
reset_counter()
exp_value = expectimax(root, True)
print(f"Expectimax Expected Utility: {exp_value:.2f} (Expected: 7.67)")
print("Notice how the expected value is much higher than 3. The agent is now willing to take risks!")

Expectimax Expected Utility: 7.67 (Expected: 7.67)
Notice how the expected value is much higher than 3. The agent is now willing to take risks!


## Task 4: Automated Negotiation
In Non-Zero-Sum environments, agents often negotiate to find a mutually beneficial agreement. The standard protocol for this is the **Alternating Offers Protocol**.

Here, we simulate two agents negotiating the price of a shared computing cluster.
* **The Seller** wants a high price (Max: \$1000) but will not accept below \$600 (Reservation Price).
* **The Buyer** wants a low price (Min: \$500) but will not pay more than \$900 (Reservation Price).

They will use **Time-Dependent Agent (TDA)** strategies.
The target utility threshold drops as time runs out according to this formula:
$$ Price(t) = Min + (Max - Min) \times (1 - (\frac{t}{T_{max}})^\beta) $$

*   If $\beta > 1$: **Boulware Strategy** (Stubborn, concedes at the last minute).
*   If $\beta < 1$: **Conceder Strategy** (Concedes quickly to secure a fast deal).

In [5]:
def calculate_offer(t, t_max, beta, min_price, max_price, is_seller):
    """
    TODO 4.1: Implement the Time-Dependent concession formula.
    """
    # 1. Calculate the time factor: (t / t_max) ^ beta
    # --- YOUR CODE HERE ---
    time_factor = (t / t_max) ** beta
    # ----------------------

    # 2. Calculate the offer
    if is_seller:
        # Seller starts high and drops down to min_price
        offer = min_price + (max_price - min_price) * (1 - time_factor)
    else:
        # Buyer starts low and climbs up to max_price
        # Notice we invert the logic because the buyer's utility goes the other way
        offer = min_price + (max_price - min_price) * time_factor

    return offer

def simulate_negotiation(seller_beta, buyer_beta):
    t_max = 20 # 20 rounds of negotiation
    seller_min, seller_max = 600, 1000
    buyer_min, buyer_max = 500, 900

    print(f"--- Starting Negotiation (Seller Beta: {seller_beta}, Buyer Beta: {buyer_beta}) ---")

    for t in range(1, t_max + 1):
        # 1. Seller makes an offer
        seller_offer = calculate_offer(t, t_max, seller_beta, seller_min, seller_max, is_seller=True)

        # 2. Buyer makes an offer
        buyer_offer = calculate_offer(t, t_max, buyer_beta, buyer_min, buyer_max, is_seller=False)

        print(f"Round {t:02d} | Seller asks: ${seller_offer:.2f} | Buyer offers: ${buyer_offer:.2f}")

        # 3. Check for Agreement (If buyer is willing to pay more than seller is asking)
        if buyer_offer >= seller_offer:
            settlement_price = (seller_offer + buyer_offer) / 2
            print(f">>> AGREEMENT REACHED at Round {t}! Final Price: ${settlement_price:.2f}\n")
            return

    print(">>> NEGOTIATION FAILED. Agents walked away.\n")

# --- TEST YOUR CODE ---
# Scenario A: Two stubborn Boulware agents (beta = 5)
simulate_negotiation(seller_beta=5, buyer_beta=5)

# Scenario B: A stubborn Seller (beta = 5) vs a Conceder Buyer (beta = 0.2)
simulate_negotiation(seller_beta=5, buyer_beta=0.2)

--- Starting Negotiation (Seller Beta: 5, Buyer Beta: 5) ---
Round 01 | Seller asks: $1000.00 | Buyer offers: $500.00
Round 02 | Seller asks: $1000.00 | Buyer offers: $500.00
Round 03 | Seller asks: $999.97 | Buyer offers: $500.03
Round 04 | Seller asks: $999.87 | Buyer offers: $500.13
Round 05 | Seller asks: $999.61 | Buyer offers: $500.39
Round 06 | Seller asks: $999.03 | Buyer offers: $500.97
Round 07 | Seller asks: $997.90 | Buyer offers: $502.10
Round 08 | Seller asks: $995.90 | Buyer offers: $504.10
Round 09 | Seller asks: $992.62 | Buyer offers: $507.38
Round 10 | Seller asks: $987.50 | Buyer offers: $512.50
Round 11 | Seller asks: $979.87 | Buyer offers: $520.13
Round 12 | Seller asks: $968.90 | Buyer offers: $531.10
Round 13 | Seller asks: $953.59 | Buyer offers: $546.41
Round 14 | Seller asks: $932.77 | Buyer offers: $567.23
Round 15 | Seller asks: $905.08 | Buyer offers: $594.92
Round 16 | Seller asks: $868.93 | Buyer offers: $631.07
Round 17 | Seller asks: $822.52 | Buyer o

### Negotiation Analysis
Run the cell above. You should observe:
1. **Scenario A (Boulware vs Boulware):** Both agents refuse to budge until the final few rounds. They barely reach an agreement at Round 18.
2. **Scenario B (Boulware vs Conceder):** The Conceder buyer rapidly increases its offer to secure a deal, while the Boulware seller holds firm. The agreement happens much faster (Round 7), but the final price heavily favors the stubborn seller.

This demonstrates why navigating Non-Zero-Sum environments requires analyzing the opponent's mathematical strategy to avoid exploitation.

**My observed results (with the code above):**
* Scenario A (Boulware vs Boulware) reached agreement at **Round 19** (final price \$750.00).
* Scenario B (Boulware seller vs Conceder buyer) reached agreement at **Round 16** (final price \$875.73).
* The qualitative conclusion holds: the Conceder buyer pays more (\$875.73 vs \$750.00), so the stubborn seller gains from the buyer's early concessions.
* Alpha-Beta evaluated **11** nodes (not 9): only L5 and L6 are pruned. Under M2 the first leaf (2) is already <= alpha (3), so the rest of M2 is cut; under M3 the 2 is found only at the last child, so nothing is left to prune.


## Task 5: Applying Frameworks - NegMAS
While building algorithms from scratch is essential for understanding the math, modern AI research relies on standardized libraries to handle environment mechanics and benchmarking.

For automated negotiation, the standard Python framework is **NegMAS (Negotiation Multi-Agent System)**. It natively supports the **Alternating Offers Protocol** and comes pre-loaded with baseline algorithms like the Boulware and Conceder strategies.

In this final task, we will replicate the negotiation from Task 4 using just a few lines of NegMAS code.

*Note: You may need to run `!pip install negmas` in a cell before running this code.*

In [6]:
!pip install negmas

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [7]:
# Note for Instructor: Remind students to pip install negmas if testing locally.
from negmas.sao import SAOMechanism
from negmas.sao.negotiators import BoulwareTBNegotiator, ConcederTBNegotiator
from negmas.preferences import LinearUtilityFunction
from negmas.outcomes import make_issue

def run_negmas_simulation():
    # 1. Define the Issue
    # Use a 3-value tuple (min, max, step) to define a discrete range issue
    price_issue = make_issue(name="price", values=(500, 1000, 1))
    issues = [price_issue]

    # 2. Define the Mechanism
    # Pass the formalized issues into the Sequential Alternating Offers mechanism
    mechanism = SAOMechanism(n_steps=20, issues=issues)

    # 3. Create the Agents using native Time-Based strategies
    seller = BoulwareTBNegotiator(name="Stubborn Seller")
    buyer = ConcederTBNegotiator(name="Desperate Buyer")

    # 4. Assign Utility Functions
    # The LinearUtilityFunction MUST know what issues it is weighing.
    # Seller wants a high price (+1 weight), Buyer wants a low price (-1 weight).
    seller_uf = LinearUtilityFunction(weights={"price": 1}, issues=issues)
    buyer_uf = LinearUtilityFunction(weights={"price": -1}, issues=issues)

    # 5. Add agents to the mechanism
    mechanism.add(seller, ufun=seller_uf)
    mechanism.add(buyer, ufun=buyer_uf)

    # 6. Run the negotiation
    state = mechanism.run()

    # 7. Output the Results
    print(f"--- NegMAS Simulation Complete ---")
    if state.agreement:
        print(f"Agreement Reached at Step {state.step}!")
        print(f"Final Outcome: {state.agreement}")
    else:
        print("Negotiation Failed (Walked Away).")

# --- TEST YOUR CODE ---
try:
    run_negmas_simulation()
except ImportError:
    print("NegMAS is not installed in this environment.")

--- NegMAS Simulation Complete ---
Agreement Reached at Step 13!
Final Outcome: (934,)


### Why Frameworks Matter
By using NegMAS, we didn't have to manually write the math for the time-dependent concession curves or manage the loop of alternating offers.

If you are developing a cutting-edge Deep Reinforcement Learning negotiator for a competition like **ANAC** or the **SCML (Supply Chain Management League)**, you plug your custom agent directly into a mechanism like this. This guarantees your agent plays by the exact same rules as the standard baseline agents (like `AspirationNegotiator`), ensuring your research results are valid and reproducible.